In [ ]:
import json
import csv
import os

def convert_jsonl_folder_to_csv(input_folder_path, csv_file_path):
    """
    Converts all JSONL files in a specified folder to a single CSV file,
    extracting English data based on specific keys, setting 'is_few_shot' to False,
    and using the input filename (without extension) as the 'Subject'.

    Args:
        input_folder_path (str): The path to the input folder containing JSONL files.
        csv_file_path (str): The path to the output CSV file.
    """
    # Define the headers for the CSV file
    headers = [
        "#", "ID", "Source", "Country", "Group", "Subject", "Level",
        "Question", "BackStory", "Context", "Answer Key", "Option 1",
        "Option 2", "Option 3", "Option 4", "Option 5", "is_few_shot"
    ]

    # Define the mapping from JSON keys to CSV headers
    # "Subject" is no longer mapped here, it will be set from the filename.
    key_mapping = {
        "序号\nINDEX": ["#", "ID"],
        "能力\nABILITY": ["Source"], # Now only maps to Source
        "STORY": ["BackStory"],
        "QUESTION": ["Question"],
        "OPTION-A": ["Option 1"],
        "OPTION-B": ["Option 2"],
        "OPTION-C": ["Option 3"],
        "OPTION-D": ["Option 4"],
        "答案\nANSWER": ["Answer Key"]
    }

    # Columns that are not present in the JSONL and will be left blank initially
    blank_columns = ["Country", "Group", "Level", "Context", "Option 5"]

    processed_files = 0
    total_lines_processed = 0

    try:
        # Open the CSV file for writing
        with open(csv_file_path, 'w', newline='', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            # Write the header row only once
            writer.writerow(headers)

            # Iterate through all files in the input folder
            for filename in os.listdir(input_folder_path):
                # Process only files ending with .jsonl
                if filename.lower().endswith('.jsonl'):
                    jsonl_file_path = os.path.join(input_folder_path, filename)
                    print(f"Processing file: {jsonl_file_path}...")
                    processed_files += 1

                    # *** MODIFICATION START ***
                    # Get the filename without the extension
                    subject_name, _ = os.path.splitext(filename)
                    # *** MODIFICATION END ***

                    try:
                        # Open the current JSONL file for reading
                        with open(jsonl_file_path, 'r', encoding='utf-8') as infile:
                            # Process each line in the JSONL file
                            for line_number, line in enumerate(infile):
                                try:
                                    # Parse the JSON object from the current line
                                    data = json.loads(line.strip())
                                    total_lines_processed += 1

                                    # Create a dictionary for the CSV row, initialized blank
                                    csv_row_dict = {header: "" for header in headers}

                                    # Explicitly set 'is_few_shot' to False
                                    csv_row_dict["is_few_shot"] = False

                                    # Set 'Subject' to the filename without extension
                                    csv_row_dict["Subject"] = subject_name # Use the modified name

                                    # Populate the dictionary using the key mapping
                                    for json_key, csv_headers in key_mapping.items():
                                        if json_key in data:
                                            value = data[json_key]
                                            for csv_header in csv_headers:
                                                # Ensure we don't overwrite explicitly set values like Subject or is_few_shot
                                                if csv_header not in ["Subject", "is_few_shot"]:
                                                    csv_row_dict[csv_header] = value

                                    # Create the row list in the correct order
                                    csv_row = [csv_row_dict[header] for header in headers]
                                    # Write the row to the CSV file
                                    writer.writerow(csv_row)

                                except json.JSONDecodeError:
                                    print(f"Error decoding JSON on line {line_number + 1} in {filename}: {line.strip()}")
                                except Exception as e:
                                    print(f"An error occurred processing line {line_number + 1} in {filename}: {e}")
                    except Exception as e:
                         print(f"Error reading file {jsonl_file_path}: {e}")

        if processed_files > 0:
            print(f"\nSuccessfully processed {processed_files} JSONL file(s) and {total_lines_processed} lines.")
            print(f"Output saved to '{csv_file_path}'")
        else:
            print(f"No .jsonl files found in the '{input_folder_path}' directory.")

    except FileNotFoundError:
        print(f"Error: Input folder not found at '{input_folder_path}'")
    except Exception as e:
        print(f"An unexpected error occurred during CSV writing: {e}")

# --- Script Execution ---
if __name__ == "__main__":
    # Define the input folder and output file paths
    # *** CHANGE THESE PATHS AS NEEDED ***
    input_folder = 'data'
    output_file = 'output.csv'

    # Check if the input folder exists before running the conversion
    if os.path.isdir(input_folder):
        convert_jsonl_folder_to_csv(input_folder, output_file)
    else:
        # Create a dummy input folder and file for demonstration if it doesn't exist
        print(f"Input folder '{input_folder}' not found. Creating a dummy folder and file for demonstration.")
        dummy_file_path = os.path.join(input_folder, 'dummy_input.jsonl')
        dummy_data = [
             {"能力\nABILITY": "Intention: Intentions explanations", "序号\nINDEX": 1, "故事": "小红和小芳...", "问题": "小红为什么...", "选项A": "选项A中文", "选项B": "选项B中文", "选项C": "选项C中文", "选项D": "选项D中文", "STORY": "Xiao Hong and Xiao Fang watch other children play...", "QUESTION": "Why does Xiao Hong smile at Xiao Fang?", "OPTION-A": "Xiao Hong smiles because she sees something interesting...", "OPTION-B": "Xiao Hong smiles because she remembers a joke...", "OPTION-C": "Xiao Hong smiles because she thinks it is fun...", "OPTION-D": "Xiao Hong smiles at Xiao Fang to suggest...", "答案\nANSWER": "D"},
             {"能力\nABILITY": "Belief: False belief", "序号\nINDEX": 2, "故事": "小明把饼干...", "问题": "小明会去哪里...", "选项A": "红色盒子", "选项B": "蓝色盒子", "选项C": "桌子上", "选项D": "问妈妈", "STORY": "Xiao Ming puts the cookies in the blue box...", "QUESTION": "Where will Xiao Ming look for the cookies?", "OPTION-A": "Red box", "OPTION-B": "Blue box", "OPTION-C": "On the table", "OPTION-D": "Ask mom", "答案\nANSWER": "B"}
        ]
        try:
            os.makedirs(input_folder, exist_ok=True) # Create folder if it doesn't exist
            with open(dummy_file_path, 'w', encoding='utf-8') as f:
                for item in dummy_data:
                    # Use ensure_ascii=False to write Chinese characters correctly if needed for demo
                    f.write(json.dumps(item, ensure_ascii=False) + '\n')
            print(f"Dummy folder '{input_folder}' and file '{dummy_file_path}' created.")
            print("Please populate the 'data' folder with your actual .jsonl files and run the script again.")
        except Exception as e:
             print(f"Could not create dummy input folder/file: {e}")


Processing file: data/Ambiguous Story Task.jsonl...
Processing file: data/Scalar Implicature Test.jsonl...
Processing file: data/Hinting Task Test.jsonl...
Processing file: data/Faux-pas Recognition Test.jsonl...
Processing file: data/Knowledge-Pretend Play Links.jsonl...
Processing file: data/Discrepant Emotions.jsonl...
Processing file: data/Multiple Desires.jsonl...
Processing file: data/False Belief Task.jsonl...
Processing file: data/Knowledge-Attention Links.jsonl...
Processing file: data/Discrepant Intentions.jsonl...
Processing file: data/Strange Story Task.jsonl...
Processing file: data/Moral Emotions.jsonl...
Processing file: data/Hidden Emotions.jsonl...
Processing file: data/Emotion Regulation.jsonl...
Processing file: data/Completion of Failed Actions.jsonl...
Processing file: data/Percepts-Knowledge Links.jsonl...
Processing file: data/Unexpected Outcome Test.jsonl...
Processing file: data/Prediction of Actions.jsonl...
Processing file: data/Persuasion Story Task.jsonl...